# Шаг 4. Baseline-модели: GlobalMean и PopularityRecommender

## Цели ноутбука
1. **Реализовать модуль метрик `src/metrics.py`** — единый источник истины для всех последующих ноутбуков.
2. **Обучить и оценить два baseline'а:**
   - `GlobalMean` — предсказывает глобальное среднее для RMSE-baseline.
   - `PopularityRecommender` — IMDb-style Bayesian average для топ-N baseline.

Все последующие модели (SVD, KNN, нейросети) **обязаны превзойти** эти baseline'ы, иначе они не имеют смысла.

## 0. Импорты и настройки

In [ ]:
import sys
sys.path.append('..')

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import SEED, set_seeds
from src.data_io import load_splits, load_features, load_id_maps
from src.metrics import (
    rmse, mae,
    evaluate_rating_prediction, evaluate_topn,
    build_ground_truth,
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
set_seeds()

MODELS_DIR    = Path('../models')
PROCESSED_DIR = Path('../data/processed')

print(f"SEED = {SEED}")

## 1. Загрузка артефактов из шага 3

Загружаем train/val/test сплиты и обогащённые данные о фильмах.

In [ ]:
splits          = load_splits()
train, val, test = splits['train'], splits['val'], splits['test']
features        = load_features()
movies_enriched = features['movies_enriched']
maps            = load_id_maps()

print(f"train: {len(train):,} | val: {len(val):,} | test: {len(test):,}")
print(f"Уникальных пользователей (train): {train['userId'].nunique()}")
print(f"Уникальных фильмов (train):       {train['movieId'].nunique()}")
print(f"Диапазон рейтингов: {train['rating'].min()} – {train['rating'].max()}")

## 2. Модель 1 — GlobalMean (baseline для RMSE)

GlobalMean — самый тривиальный baseline. Предсказываем глобальное среднее
train-рейтинга для любой пары (user, item).
**Любая разумная модель должна его побеждать по RMSE.**

In [ ]:
class GlobalMeanModel:
    """Тривиальный baseline: предсказывает глобальное среднее train-рейтинга."""

    def __init__(self):
        self.global_mean_: float | None = None

    def fit(self, train_df: pd.DataFrame,
            rating_col: str = 'rating') -> 'GlobalMeanModel':
        self.global_mean_ = float(train_df[rating_col].mean())
        return self

    def predict(self, user_ids: np.ndarray,
                movie_ids: np.ndarray) -> np.ndarray:
        if self.global_mean_ is None:
            raise RuntimeError('Модель не обучена')
        return np.full(len(user_ids), self.global_mean_, dtype=np.float64)


global_mean_model = GlobalMeanModel().fit(train)
print(f'Глобальное среднее train: {global_mean_model.global_mean_:.4f}')

In [ ]:
# Оценка GlobalMean на val и test
val_preds  = global_mean_model.predict(val['userId'].values, val['movieId'].values)
test_preds = global_mean_model.predict(test['userId'].values, test['movieId'].values)

global_mean_metrics = {
    'val':  evaluate_rating_prediction(val['rating'].values, val_preds),
    'test': evaluate_rating_prediction(test['rating'].values, test_preds),
}
print(json.dumps(global_mean_metrics, indent=2))

## 3. Модель 2 — PopularityRecommender (baseline для топ-N)

**Байесовское среднее (IMDb-style):**

```
score(i) = (n_i / (n_i + m)) * mean_i + (m / (n_i + m)) * C
```

где:
- `n_i` — число оценок фильма в train
- `mean_i` — средний рейтинг фильма
- `C` — глобальное среднее (prior)
- `m` — параметр сглаживания (подбираем через grid search)

При малом `n_i` оценка прижимается к `C`; при большом `n_i` — к `mean_i`.
Один и тот же рейтинг для всех пользователей, но с фильтрацией уже просмотренных.

In [ ]:
class PopularityRecommender:
    """Топ-N рекомендатель на основе сглаженной популярности (Bayesian average).

    Один и тот же топ-N для всех пользователей, но с исключением уже оценённых.
    """

    def __init__(self, m: float = 10.0):
        self.m = m
        self.global_mean_: float | None = None
        self.scores_: pd.Series | None = None   # index=movieId, value=score
        self.user_seen_: dict[int, set] | None = None

    def fit(self, train_df: pd.DataFrame,
            user_col: str = 'userId', item_col: str = 'movieId',
            rating_col: str = 'rating') -> 'PopularityRecommender':
        self.global_mean_ = float(train_df[rating_col].mean())
        agg = train_df.groupby(item_col)[rating_col].agg(['count', 'mean'])
        n = agg['count']
        mu = agg['mean']
        C = self.global_mean_
        self.scores_ = (n / (n + self.m)) * mu + (self.m / (n + self.m)) * C
        self.scores_ = self.scores_.sort_values(ascending=False)
        self.user_seen_ = (
            train_df.groupby(user_col)[item_col].apply(set).to_dict()
        )
        return self

    def recommend(self, user_ids: list[int], k: int = 10,
                  exclude_seen: bool = True) -> dict[int, list]:
        if self.scores_ is None:
            raise RuntimeError('Модель не обучена')
        ranked_items = self.scores_.index.values
        result: dict[int, list] = {}
        for u in user_ids:
            if exclude_seen:
                seen = self.user_seen_.get(u, set())
                recs = [item for item in ranked_items if item not in seen][:k]
            else:
                recs = list(ranked_items[:k])
            result[u] = recs
        return result

## 4. Подбор параметра m через grid search на val

У popularity-модели один гиперпараметр — `m`. Пространство параметров крошечное,
поэтому Optuna избыточна — используем простой grid search по NDCG@10 на val.
(Это эквивалентно `optuna.create_study(direction='maximize')` с категориальным параметром.)

In [ ]:
val_ground_truth = build_ground_truth(val, relevance_threshold=4.0)
val_users        = list(val_ground_truth.keys())
all_movies       = train['movieId'].unique()

m_grid = [0.0, 1.0, 5.0, 10.0, 20.0, 50.0, 100.0, 200.0]
grid_results = []

for m in m_grid:
    model = PopularityRecommender(m=m).fit(train)
    recs  = model.recommend(val_users, k=10, exclude_seen=True)
    metrics = evaluate_topn(recs, val_ground_truth, ks=(10,), all_items=all_movies)
    grid_results.append({'m': m, **metrics})

grid_df = pd.DataFrame(grid_results)
display(grid_df.round(4))

best_m = float(grid_df.loc[grid_df['ndcg@10'].idxmax(), 'm'])
print(f'\nЛучшее m по NDCG@10 на val: {best_m}')

In [ ]:
# График: зависимость метрик от m
fig, ax = plt.subplots(figsize=(9, 4))

for metric, color, marker in [
    ('ndcg@10',      'steelblue',  'o'),
    ('precision@10', 'darkorange', 's'),
    ('recall@10',    'seagreen',   '^'),
]:
    ax.plot(range(len(m_grid)), grid_df[metric],
            label=metric, color=color, marker=marker, linewidth=2)

ax.set_xticks(range(len(m_grid)))
ax.set_xticklabels([str(m) for m in m_grid])
ax.set_xlabel('Параметр m (Bayesian prior)')
ax.set_ylabel('Значение метрики')
ax.set_title('Зависимость метрик от параметра m (val)')
ax.legend()
ax.axvline(x=m_grid.index(best_m), color='gray', linestyle='--', alpha=0.6,
           label=f'best m={best_m}')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Финальная модель и оценка на test

Финальные модели переобучаются на `train + val`, затем однократно оцениваются на `test`.

In [ ]:
# Финальное обучение на train + val
train_val = pd.concat([train, val], ignore_index=True)

final_pop_model = PopularityRecommender(m=best_m).fit(train_val)
final_gm_model  = GlobalMeanModel().fit(train_val)

# ── GlobalMean на test ──
test_preds_gm   = final_gm_model.predict(test['userId'].values, test['movieId'].values)
gm_test_metrics = evaluate_rating_prediction(test['rating'].values, test_preds_gm)

# ── PopularityRecommender на test ──
test_ground_truth = build_ground_truth(test, relevance_threshold=4.0)
test_users        = list(test_ground_truth.keys())
test_recs         = final_pop_model.recommend(test_users, k=20, exclude_seen=True)
pop_test_metrics  = evaluate_topn(
    test_recs, test_ground_truth,
    ks=(5, 10, 20),
    all_items=train_val['movieId'].unique()
)

print('GlobalMean — test:')
print(json.dumps(gm_test_metrics, indent=2))
print('\nPopularity — test:')
print(json.dumps(pop_test_metrics, indent=2))

## 6. Анализ результатов

1. **RMSE GlobalMean** (~1.05–1.15 для MovieLens) — это «потолок плохости» по RMSE.
   Любая модель коллаборативной фильтрации должна уверенно его бить.
   GlobalMean игнорирует и личные предпочтения, и специфику фильмов.

2. **NDCG@10 Popularity** — это «потолок плохости» по ранжированию.
   Если будущая модель не побеждает Popularity по NDCG@10, она бесполезна:
   простое «порекомендовать самые популярные» оказывается лучше.

3. **Coverage Popularity** ≈ 10–20 / |all_movies| — крошечный (< 0.5%).
   Popularity рекомендует одни и те же блокбастеры всем пользователям.
   Это классическая иллюстрация trade-off accuracy vs diversity:
   высокий хит-рейт, но нулевое разнообразие.

4. **Hit Rate@10 Popularity** — показывает, какой процент пользователей
   получает хотя бы один релевантный фильм в топ-10. Для Popularity
   это достаточно высокое значение благодаря концентрации на универсальных хитах.

In [ ]:
# Топ-10 «самых популярных» фильмов
top10_ids    = final_pop_model.scores_.head(10).index
top10_titles = movies_enriched.set_index('movieId').loc[top10_ids, 'title']
top10_scores = final_pop_model.scores_.head(10).values
top10_df = pd.DataFrame({'title': top10_titles.values, 'score': top10_scores.round(4)})
display(top10_df)

# Bar chart
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(range(len(top10_df)), top10_df['score'],
        color='steelblue', edgecolor='white')
ax.set_yticks(range(len(top10_df)))
ax.set_yticklabels(top10_df['title'], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Bayesian score')
ax.set_title('Топ-10 фильмов по Popularity score')
plt.tight_layout()
plt.show()

In [ ]:
# Сводная таблица метрик
summary_rows = []
for metric_name, val_score in grid_df[grid_df['m'] == best_m].iloc[0].items():
    if metric_name == 'm':
        continue
    summary_rows.append({'Метрика': metric_name, 'Val (best m)': round(float(val_score), 4)})
summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.merge(
    pd.DataFrame([{'Метрика': k.replace('@10', '@10'), 'Test': round(v, 4)}
                  for k, v in pop_test_metrics.items() if '@10' in k]),
    on='Метрика', how='left'
)
print("Метрики PopularityRecommender (k=10):")
display(summary_df)

## 7. Сохранение артефактов

In [ ]:
joblib.dump(final_gm_model,  MODELS_DIR / 'global_mean_model.pkl')
joblib.dump(final_pop_model, MODELS_DIR / 'popularity_model.pkl')

popularity_params = {
    'random_state':        SEED,
    'best_m':              best_m,
    'm_grid':              m_grid,
    'relevance_threshold': 4.0,
    'final_train_strategy': 'train + val concatenated',
}
with open(MODELS_DIR / 'popularity_params.json', 'w', encoding='utf-8') as f:
    json.dump(popularity_params, f, ensure_ascii=False, indent=2)

popularity_metrics = {
    'global_mean': {
        'val':  global_mean_metrics['val'],
        'test': gm_test_metrics,
    },
    'popularity': {
        'val_grid_search': grid_df.round(6).to_dict(orient='records'),
        'test':            pop_test_metrics,
    },
    'meta': {
        'k_values':            [5, 10, 20],
        'relevance_threshold': 4.0,
        'best_m':              best_m,
    },
}
with open(MODELS_DIR / 'popularity_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(popularity_metrics, f, ensure_ascii=False, indent=2)

print("Артефакты сохранены:")
for name in ['global_mean_model.pkl', 'popularity_model.pkl',
             'popularity_params.json', 'popularity_metrics.json']:
    p = MODELS_DIR / name
    print(f"  {name}: {p.stat().st_size} байт")

## 8. Финальные проверки

In [ ]:
# Файлы существуют
required_files = [
    MODELS_DIR / 'global_mean_model.pkl',
    MODELS_DIR / 'popularity_model.pkl',
    MODELS_DIR / 'popularity_params.json',
    MODELS_DIR / 'popularity_metrics.json',
]
for p in required_files:
    assert p.exists(), f'Файл не найден: {p}'

# Round-trip моделей
gm_loaded  = joblib.load(MODELS_DIR / 'global_mean_model.pkl')
pop_loaded = joblib.load(MODELS_DIR / 'popularity_model.pkl')
assert abs(gm_loaded.global_mean_ - final_gm_model.global_mean_) < 1e-9
assert pop_loaded.m == best_m
assert pop_loaded.scores_ is not None and len(pop_loaded.scores_) > 0

# Sanity-границы метрик
assert 0.5 < gm_test_metrics['rmse'] < 2.5, f"RMSE GlobalMean подозрителен: {gm_test_metrics['rmse']}"
assert 0.0 <= pop_test_metrics['ndcg@10'] <= 1.0
assert 0.0 <= pop_test_metrics['precision@10'] <= 1.0
assert 0.0 <= pop_test_metrics['recall@10'] <= 1.0

# Проверка рекомендаций: нет дубликатов и нет просмотренных
sample_user = test_users[0]
sample_recs = final_pop_model.recommend([sample_user], k=10)[sample_user]
assert len(sample_recs) == 10
assert len(set(sample_recs)) == 10, 'В рекомендациях есть дубликаты'
seen_by_user = set(train_val[train_val['userId'] == sample_user]['movieId'])
assert not (set(sample_recs) & seen_by_user), 'В рекомендациях есть уже просмотренные фильмы'

print('\u2705 Шаг 4 пройден: метрики реализованы, baseline-модели обучены и оценены')
print(f'   GlobalMean RMSE (test):  {gm_test_metrics["rmse"]:.4f}')
print(f'   Popularity NDCG@10 (test): {pop_test_metrics["ndcg@10"]:.4f}')
print(f'   Popularity Coverage@20 (test): {pop_test_metrics.get("coverage@20", 0):.4f}')